## Build Constructors Dimension

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01-Environment_Config

In [0]:
%run ../00-common/04-Gold_Helpers

In [0]:
constructors_table = f"{catalog_name}.{silver_schema}.constructors"
nationality_table = f"{catalog_name}.{gold_schema}.ref_nationality_region"

gold_table = f"{catalog_name}.{gold_schema}.dim_constructors"

In [0]:
from pyspark.sql import functions as f

In [0]:
constructors_df = spark.read.table(constructors_table).filter((f.col("batch_id") == v_batch_id))

nationality_region_df = spark.read.table(nationality_table)

In [0]:
joined_df = constructors_df.join(
    nationality_region_df, constructors_df.nationality == nationality_region_df.nationality, how="left"
).select(
    constructors_df.constructor_Id, 
    constructors_df.constructor_name, 
    constructors_df.nationality, 
    nationality_region_df.Region.alias("nationality_region")
    )

In [0]:
columns = [
    "constructor_name",
    "nationality",
    "nationality_region"
]

write_to_gold(joined_df, gold_table, "trg.constructor_Id = src.constructor_Id", columns)